# [실습] 나만의 캐릭터 말투 Fine-Tuning

앞서 조선시대 신하 말투를 학습시켜 봤습니다.  
이번에는 **나만의 캐릭터 말투** 데이터를 직접 만들고 fine-tuning 해봅니다.

| 항목 | 데모에서 한 것 | 이번 실습 |
|------|----------------|----------|
| 데이터 | 조선시대 말투 200개 (제공) | 나만의 캐릭터 말투 30개 이상 (직접 제작) |
| 모델 | Qwen2.5-1.5B-Instruct | Qwen2.5-1.5B-Instruct (동일) |
| 방법 | QLoRA (Unsloth) | QLoRA (Unsloth) (동일) |

### 캐릭터 아이디어 예시

| 캐릭터 | 말투 특징 | 예시 |
|--------|----------|------|
| 부산 사투리 | ~다 아이가, ~하이소 | "밥 묵었나? 안 묵었으면 같이 가자 아이가" |
| 츤데레 | 관심 없는 척, ~별로야 | "딱히 네가 걱정돼서 그런 건 아니거든..." |
| 할머니 | ~하렴, ~구나 | "어이고 우리 손주, 밥은 먹었니? 이것도 먹어보렴" |
| 해적 | ~다 이놈아, 보물 관련 비유 | "크하하! 그건 보물섬만큼 귀한 정보다 이놈아!" |
| 군인 | ~입니다, 간결함 | "보고드립니다. 점심 메뉴는 제육볶음입니다." |

> **자유롭게** 원하는 캐릭터를 정하세요. 위 예시가 아닌 자기만의 캐릭터도 좋습니다!

---
## 1단계: 패키지 설치

Unsloth와 pyarrow를 설치하세요. 데모와 동일합니다.

> `%%capture`를 셀 맨 위에 쓰면 설치 로그가 숨겨져서 깔끔합니다.

---
## 2단계: 모델 로드

Unsloth의 `FastLanguageModel.from_pretrained()`으로 모델을 불러오세요.

설정:
- `model_name`: `"unsloth/Qwen2.5-1.5B-Instruct"`
- `max_seq_length`: `1024`
- `load_in_4bit`: `True` (4비트 양자화로 메모리 절약)

이 함수는 `model`과 `tokenizer` 두 개를 반환합니다.

---
## 3단계: LoRA 설정

`FastLanguageModel.get_peft_model()`로 LoRA를 적용하세요.

설정:
- `r`: `16` (LoRA rank — 높을수록 표현력 증가, 메모리 증가)
- `target_modules`: attention과 FFN의 projection 레이어들
  ```python
  ["q_proj", "k_proj", "v_proj", "o_proj",
   "gate_proj", "up_proj", "down_proj"]
  ```
- `lora_alpha`: `16`
- `lora_dropout`: `0`
- `bias`: `"none"`
- `use_gradient_checkpointing`: `"unsloth"` (메모리 최적화)

---
## 4단계: 나만의 학습 데이터 만들기 ⭐

**이 단계가 이번 실습의 핵심입니다!**

아래 형식으로 최소 **30개 이상**의 대화 데이터를 만드세요:

```python
data = [
    {
        "instruction": "일상적인 질문 (평범한 말투)",
        "output": "캐릭터 말투로 된 답변"
    },
    # ... 30개 이상
]
```

### 잘 만드는 팁

1. **다양한 주제를 다루세요** — 인사, 날씨, 음식, 고민 상담, 추천, 감정 표현 등
2. **말투를 일관되게** — 같은 캐릭터가 어떤 질문에도 그 말투를 유지해야 합니다
3. **답변 길이를 다양하게** — 짧은 답변, 긴 답변 섞기

### 데이터를 JSON으로 저장하기

데이터를 만든 뒤, JSON 파일로 저장하세요:
```python
import json

with open("my_character.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
```

> **또는** 이미 만들어둔 JSON 파일이 있다면 Colab에 업로드해도 됩니다.  
> 형식만 `[{"instruction": "...", "output": "..."}, ...]` 이면 OK입니다.

---
## 5단계: 데이터를 학습 형식으로 변환

JSON 데이터를 모델이 학습할 수 있는 형식으로 바꿔야 합니다.

### 해야 할 것:
1. JSON 파일을 읽어서 `Dataset` 객체로 변환
   ```python
   from datasets import Dataset
   dataset = Dataset.from_list(data)  # 또는 JSON 파일에서 로드
   ```

2. 각 데이터를 Qwen 채팅 템플릿 형식으로 변환하는 함수를 만드세요:
   ```python
   def format_example(example):
       messages = [
           {"role": "user",      "content": example["instruction"]},
           {"role": "assistant", "content": example["output"]},
       ]
       text = tokenizer.apply_chat_template(
           messages,
           tokenize=False,
           add_generation_prompt=False,
       )
       return {"text": text}
   ```

3. `dataset.map(format_example)`으로 변환 적용

4. 변환된 샘플 1개를 출력해서 형식이 맞는지 확인

---
## 6단계: 학습 실행

`trl`의 `SFTTrainer`로 학습을 실행하세요.

```python
from trl import SFTTrainer, SFTConfig
```

### SFTConfig 설정

| 설정 | 값 | 설명 |
|------|----|------|
| `dataset_text_field` | `"text"` | 5단계에서 만든 텍스트 컬럼 |
| `max_seq_length` | `1024` | |
| `per_device_train_batch_size` | `2` | Colab 메모리 제한 |
| `gradient_accumulation_steps` | `4` | 실질 배치 = 2×4 = 8 |
| `num_train_epochs` | `3` | 데이터가 적으니 3회 반복 |
| `learning_rate` | `2e-4` | LoRA는 보통 full fine-tuning보다 높은 lr 사용 |
| `warmup_ratio` | `0.1` | |
| `lr_scheduler_type` | `"cosine"` | |
| `fp16` | `True` | |
| `logging_steps` | `10` | |
| `output_dir` | `"./my_character_output"` | |
| `report_to` | `"none"` | |

### SFTTrainer 생성
- `model`, `tokenizer`, `train_dataset`, `args`를 넘기세요
- `trainer.train()`으로 학습 시작

> 데이터 30개 기준으로 약 1~2분이면 학습이 끝납니다.

---
## 7단계: 결과 확인

학습된 모델에 다양한 질문을 넣어서 캐릭터 말투가 잘 나오는지 확인하세요.

### 생성 함수 작성
```python
def generate(prompt, model, max_new_tokens=200):
    FastLanguageModel.for_inference(model)  # 추론 모드로 전환
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
```

### 테스트 질문
학습 데이터에 **없었던 새로운 질문**으로 테스트하세요. 예시:

```python
test_questions = [
    "오늘 점심 뭐 먹을까?",
    "요즘 너무 힘들어...",
    "주말에 뭐 하면 좋을까?",
    "코딩 공부 어떻게 시작하면 돼?",
    "비가 오는데 우산을 안 가져왔어",
]
```

각 질문에 대한 답변을 출력하고, 캐릭터 말투가 얼마나 잘 반영되었는지 확인해보세요.

---
## 8단계: 모델 저장 (선택)

학습된 LoRA 가중치를 저장하세요.

```python
model.save_pretrained("my_character_lora")
tokenizer.save_pretrained("my_character_lora")
```

구글 드라이브에 저장하고 싶다면:
```python
from google.colab import drive
drive.mount('/content/drive')
model.save_pretrained("/content/drive/MyDrive/my_character_lora")
```

---
## 도전 과제 (선택)

시간이 남으면 아래를 시도해보세요:

1. **데이터 늘리기**: 30개 → 50개 이상으로 늘리고 다시 학습해보세요. 말투가 더 자연스러워지나요?
2. **에폭 실험**: `num_train_epochs`를 5, 10으로 올려보세요. 너무 높이면 오히려 이상한 말이 반복될 수 있습니다 (과적합)
3. **system prompt 추가**: 데이터의 `messages`에 `{"role": "system", "content": "너는 OO 캐릭터야"}`를 추가하면 어떻게 달라지는지 실험해보세요